# BKMeeting AI Hub Option 1 NPU Pilots

This notebook adapts the minimal `On_device_Ai.ipynb` example into a repo-specific Qualcomm AI Hub workflow for BKMeeting.

Scope of this notebook:

- stay fully in `python-model-test`
- do not touch Android packaging
- prove compile, profile, and inference on Qualcomm AI Hub first
- run two pilots:
  - Zipformer encoder-first
  - VPCD model-session-first


## Environment Notes

Before running this notebook, make sure the current environment can already execute the local `python-model-test` bundle helpers.

Minimum practical dependencies:

- `qai-hub`
- `torch`
- `torchaudio`
- `numpy`
- local editable install of this repo if needed

The Zipformer pilot uses the existing repo feature-extraction path, so `torchaudio` must be available.


In [ ]:
!pip install qai-hub "qai-hub[torch]"

In [ ]:
from pathlib import Path
import subprocess
import sys

sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_pilots import resolve_qai_hub_api_token

API_TOKEN = resolve_qai_hub_api_token(repo_root=Path.cwd())

if not API_TOKEN:
    print("Set QAI_HUB_API_TOKEN in .env or your shell environment before running Qualcomm AI Hub configuration.")
else:
    subprocess.run(["qai-hub", "configure", "--api_token", API_TOKEN], check=True)

!qai-hub list-devices


In [ ]:
import sys
from pathlib import Path

import qai_hub as hub

sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_pilots import (
    build_compile_options,
    build_job_options,
    build_option1_runtime_config,
    build_vpcd_input_specs,
    build_vpcd_single_step_calibration_entries,
    build_vpcd_single_step_inputs,
    build_zipformer_encoder_inference_entries,
    build_zipformer_encoder_input_specs,
    coerce_inputs_for_compiled_model,
    prepare_vpcd_option1_source_model,
    prepare_zipformer_encoder_option1_source_model,
    resolve_vpcd_fp32_source_model_path,
    resolve_vpcd_pilot_source,
    resolve_zipformer_encoder_pilot_source,
    write_live_run_record,
    write_prepared_artifact_record,
    wrap_single_inference_inputs,
)


In [ ]:
DEVICE_NAME = "Samsung Galaxy S24 (Family)"
QAIRT_VERSION = None

RUNTIME_CONFIG = build_option1_runtime_config(
    device_name=DEVICE_NAME,
    qairt_version=QAIRT_VERSION,
    repo_root=Path.cwd(),
)
job_options = build_job_options(
    compute_unit=RUNTIME_CONFIG.compute_unit,
    qairt_version=RUNTIME_CONFIG.qairt_version,
)

print("device:", RUNTIME_CONFIG.device_name)
print("qairt_version:", RUNTIME_CONFIG.qairt_version)
print("artifact_root:", RUNTIME_CONFIG.artifact_root)
print("record_root:", RUNTIME_CONFIG.record_root)
print("job_options:", job_options)


## Pilot 1: Zipformer Encoder-First

This pilot targets the first ASR slice that BKMeeting wants to offload first: the encoder graph.

The current local helper now prepares a dedicated AI Hub upload artifact from the fixed-shape encoder source.

- base source: fixed-shape encoder ONNX
- upload artifact: ORT-optimized + symbolic-shape-prepared + HTP bool-slice rewrite
- local fixtures: current Zipformer bundle sample manifest
- current verified lane: direct `submit_compile_job(...)` on the prepared source model


In [ ]:
zipformer_pilot_name = "zipformer_encoder_option1"
zipformer_source = resolve_zipformer_encoder_pilot_source(RUNTIME_CONFIG.repo_root)
zipformer_source_model_path = prepare_zipformer_encoder_option1_source_model(
    zipformer_source,
    output_path=RUNTIME_CONFIG.pilot_artifact_dir(zipformer_pilot_name) / "encoder.aihub.option1.onnx",
)
zipformer_input_specs = build_zipformer_encoder_input_specs(zipformer_source)
zipformer_compile_options = build_compile_options(
    qairt_version=RUNTIME_CONFIG.qairt_version,
    input_specs=zipformer_input_specs,
)
zipformer_raw_inference_inputs = build_zipformer_encoder_inference_entries(zipformer_source)
zipformer_inference_inputs = coerce_inputs_for_compiled_model(
    zipformer_raw_inference_inputs,
    input_specs=zipformer_input_specs,
)
zipformer_prepared_record_path = write_prepared_artifact_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    source_model_path=zipformer_source.source_model_path,
    prepared_model_path=zipformer_source_model_path,
    input_specs=zipformer_input_specs,
    compile_options=zipformer_compile_options,
    run_label="latest",
)

print("zipformer base source model:", zipformer_source.source_model_path)
print("zipformer prepared upload model:", zipformer_source_model_path)
print("zipformer bundle manifest:", zipformer_source.bundle_manifest_path)
print("zipformer input specs:", zipformer_input_specs)
print("zipformer compile options:", zipformer_compile_options)
print("zipformer prepared record:", zipformer_prepared_record_path)
print({name: [value.shape for value in values] for name, values in zipformer_inference_inputs.items()})


In [ ]:
zipformer_compile_job = hub.submit_compile_job(
    model=zipformer_source_model_path,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    input_specs=zipformer_input_specs,
    options=zipformer_compile_options,
    name="bkmeeting-zipformer-encoder-precompiled-qnn-onnx",
)

print("zipformer compile job:", zipformer_compile_job.url)


In [ ]:
zipformer_target_model = zipformer_compile_job.get_target_model()
zipformer_profile_job = hub.submit_profile_job(
    model=zipformer_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    options=job_options,
    name="bkmeeting-zipformer-encoder-profile-npu",
)

zipformer_profile = zipformer_profile_job.download_profile()
zipformer_inference_job = hub.submit_inference_job(
    model=zipformer_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    inputs=zipformer_inference_inputs,
    options=job_options,
    name="bkmeeting-zipformer-encoder-inference-npu",
)
zipformer_output = zipformer_inference_job.download_output_data()
zipformer_live_record_path = write_live_run_record(
    pilot_name=zipformer_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=zipformer_compile_options,
    job_options=job_options,
    compile_job=zipformer_compile_job,
    profile_job=zipformer_profile_job,
    inference_job=zipformer_inference_job,
    output_tensors=zipformer_output,
    run_label="latest",
)

print("zipformer profile job:", zipformer_profile_job.url)
print("zipformer inference job:", zipformer_inference_job.url)
print("zipformer live record:", zipformer_live_record_path)
print("zipformer output tensors:", {name: [value.shape for value in values] for name, values in zipformer_output.items()})


## Pilot 2: VPCD Model-Session-First

This pilot targets the punctuation model session while keeping tokenization on the host side.

Important current caveats:

- prefer the repo FP32 export when available, then freeze it to the fixed bundle shapes before upload
- if the source is still QDQ after preparation, compile it directly as the pragmatic fallback lane
- compiled inference inputs must be coerced from `int64` to `int32` when `--truncate_64bit_io` is present


In [ ]:
vpcd_pilot_name = "vpcd_option1"
vpcd_source = resolve_vpcd_pilot_source(RUNTIME_CONFIG.repo_root)
vpcd_original_source_model_path = resolve_vpcd_fp32_source_model_path(vpcd_source) or vpcd_source.model_path
vpcd_prepared_source_model_path, vpcd_is_quantized_source = prepare_vpcd_option1_source_model(
    vpcd_source,
    output_path=RUNTIME_CONFIG.pilot_artifact_dir(vpcd_pilot_name) / "model.option1.onnx",
)
vpcd_input_specs = build_vpcd_input_specs(vpcd_source)
vpcd_compile_options = build_compile_options(
    qairt_version=RUNTIME_CONFIG.qairt_version,
    input_specs=vpcd_input_specs,
)
vpcd_calibration_data = build_vpcd_single_step_calibration_entries(vpcd_source, max_samples=4)
vpcd_single_step_inputs = build_vpcd_single_step_inputs(vpcd_source, sample_index=0)
vpcd_raw_inference_inputs = wrap_single_inference_inputs(vpcd_single_step_inputs)
vpcd_inference_inputs = coerce_inputs_for_compiled_model(
    vpcd_raw_inference_inputs,
    input_specs=vpcd_input_specs,
)
vpcd_prepared_record_path = write_prepared_artifact_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    source_model_path=vpcd_original_source_model_path,
    prepared_model_path=vpcd_prepared_source_model_path,
    input_specs=vpcd_input_specs,
    compile_options=vpcd_compile_options,
    run_label="latest",
)

print("vpcd source model:", vpcd_original_source_model_path)
print("vpcd prepared upload model:", vpcd_prepared_source_model_path)
print("vpcd input specs:", vpcd_input_specs)
print("vpcd compile options:", vpcd_compile_options)
print("vpcd quantized source:", vpcd_is_quantized_source)
print("vpcd prepared record:", vpcd_prepared_record_path)
print({name: [value.shape for value in values] for name, values in vpcd_inference_inputs.items()})


In [ ]:
if vpcd_is_quantized_source:
    vpcd_compile_input_model = vpcd_prepared_source_model_path
    print("VPCD source is already QDQ. Compiling directly for the current AI Hub pilot.")
else:
    vpcd_quantize_job = hub.submit_quantize_job(
        model=vpcd_prepared_source_model_path,
        calibration_data=vpcd_calibration_data,
        name="bkmeeting-vpcd-quantize",
    )
    vpcd_compile_input_model = vpcd_quantize_job.get_target_model()
    print("vpcd quantize job:", vpcd_quantize_job.url)

vpcd_compile_job = hub.submit_compile_job(
    model=vpcd_compile_input_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    input_specs=vpcd_input_specs,
    options=vpcd_compile_options,
    name="bkmeeting-vpcd-precompiled-qnn-onnx",
)

print("vpcd compile job:", vpcd_compile_job.url)


In [ ]:
vpcd_target_model = vpcd_compile_job.get_target_model()
vpcd_profile_job = hub.submit_profile_job(
    model=vpcd_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    options=job_options,
    name="bkmeeting-vpcd-profile-npu",
)

vpcd_profile = vpcd_profile_job.download_profile()
vpcd_inference_job = hub.submit_inference_job(
    model=vpcd_target_model,
    device=hub.Device(RUNTIME_CONFIG.device_name),
    inputs=vpcd_inference_inputs,
    options=job_options,
    name="bkmeeting-vpcd-inference-npu",
)
vpcd_output = vpcd_inference_job.download_output_data()
vpcd_live_record_path = write_live_run_record(
    pilot_name=vpcd_pilot_name,
    runtime_config=RUNTIME_CONFIG,
    compile_options=vpcd_compile_options,
    job_options=job_options,
    compile_job=vpcd_compile_job,
    profile_job=vpcd_profile_job,
    inference_job=vpcd_inference_job,
    output_tensors=vpcd_output,
    run_label="latest",
)

print("vpcd profile job:", vpcd_profile_job.url)
print("vpcd inference job:", vpcd_inference_job.url)
print("vpcd live record:", vpcd_live_record_path)
print("vpcd output tensors:", {name: [value.shape for value in values] for name, values in vpcd_output.items()})


## After The Notebook Runs

This notebook now leaves behind a deterministic Phase 2 evidence trail for each pilot:

- prepared upload artifact under `build/aihub/<pilot>/`
- prepared artifact record under `build/aihub/records/<pilot>/prepared-artifact-latest.json`
- live run record under `build/aihub/records/<pilot>/live-run-latest.json`
- AI Hub job URLs printed in the execution cells

Phase 2 stops at reproducibility and handoff quality. The next phase starts only after these records are present and reviewed.


In [ ]:
print("runtime record root:", RUNTIME_CONFIG.record_root)
print("zipformer prepared record:", zipformer_prepared_record_path)
print("zipformer live record:", zipformer_live_record_path)
print("vpcd prepared record:", vpcd_prepared_record_path)
print("vpcd live record:", vpcd_live_record_path)
